In [1]:
!pip install -q transformers torchvision albumentations

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
!git clone https://github.com/tangsanli5201/DeepPCB.git

fatal: destination path 'DeepPCB' already exists and is not an empty directory.


In [4]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset


class DeepPCBDataset(Dataset):
    def __init__(self, root_dir, split_file):
        self.root_dir = root_dir

        with open(split_file, "r") as f:
            self.samples = [line.strip().split() for line in f.readlines()]

        self.classes = ["defect"]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_rel_path, ann_rel_path = self.samples[idx]

        # Usar imagen _test.jpg
        img_path = os.path.join(
            self.root_dir,
            img_rel_path.replace(".jpg", "_test.jpg")
        )
        ann_path = os.path.join(self.root_dir, ann_rel_path)

        image = Image.open(img_path).convert("RGB")
        img_w, img_h = image.size

        boxes = []
        labels = []

        if os.path.exists(ann_path):
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split()

                    if len(parts) < 5:
                        continue

                    nums = list(map(float, parts[1:5]))

                    # Detectar formato automáticamente
                    if max(nums) <= 1.0:
                        # YOLO (normalizado)
                        x_c, y_c, w, h = nums

                        xmin = (x_c - w / 2)
                        ymin = (y_c - h / 2)
                        xmax = (x_c + w / 2)
                        ymax = (y_c + h / 2)

                    else:
                        # Formato absoluto → convertir a normalizado
                        xmin, ymin, xmax, ymax = nums

                        xmin /= img_w
                        xmax /= img_w
                        ymin /= img_h
                        ymax /= img_h

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(0)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "class_labels": torch.tensor(labels, dtype=torch.int64)
        }

        return image, target

In [5]:
ROOT_DIR = "/content/DeepPCB/PCBData"
SPLIT_FILE = "/content/DeepPCB/PCBData/trainval.txt"

dataset = DeepPCBDataset(ROOT_DIR, SPLIT_FILE)

print("Samples:", len(dataset))
print("Clases:", dataset.classes)

Samples: 1000
Clases: ['defect']


In [6]:
img, target = dataset[0]

print(img.size)
print(target)

(640, 640)
{'boxes': tensor([[0.6156, 0.6797, 0.6594, 0.0047],
        [0.5984, 0.4984, 0.6516, 0.0047],
        [0.2547, 0.0562, 0.2984, 0.0063],
        [0.2359, 0.4219, 0.2844, 0.0078],
        [0.8109, 0.5688, 0.8484, 0.0094],
        [0.7188, 0.7844, 0.7516, 0.0063]]), 'class_labels': tensor([0, 0, 0, 0, 0, 0])}


In [7]:
import torch
from torch.utils.data import random_split

# Fijar semilla para reproducibilidad
torch.manual_seed(42)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 800
Test size: 200


In [11]:
import torch
from transformers import DetrForObjectDetection, DetrImageProcessor

# Processor
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

# Mapeo de clases (6 defectos)
id2label = {
    0: "open",
    1: "short",
    2: "mousebite",
    3: "spur",
    4: "copper",
    5: "pin-hole",
}

label2id = {v: k for k, v in id2label.items()}

# Modelo con 6 clases
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Modelo listo en:", device)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                                         | Status     |                                                                                      
----------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                      
model.backbone.conv_encoder.model.layer3.0.downsam

Modelo listo en: cuda


In [12]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]

    encoding = processor(images=images, return_tensors="pt")

    return {
        "pixel_values": encoding["pixel_values"],
        "labels": targets
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

print("Collate y loaders listos")

Collate y loaders listos


In [ ]:
import torch

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:

        pixel_values = batch["pixel_values"].to(device)

        labels = [
            {k: v.to(device) for k, v in t.items()}
            for t in batch["labels"]
        ]

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f}")

Epoch 1/20 - Loss: 1371.9079
Epoch 2/20 - Loss: 1010.4942
Epoch 3/20 - Loss: 840.8579
Epoch 4/20 - Loss: 804.4373
Epoch 5/20 - Loss: 770.5043
Epoch 6/20 - Loss: 755.2811
Epoch 7/20 - Loss: 751.3159
Epoch 8/20 - Loss: 731.8149
Epoch 9/20 - Loss: 734.5597
Epoch 10/20 - Loss: 720.9670
Epoch 11/20 - Loss: 722.2559
Epoch 12/20 - Loss: 709.9603
Epoch 13/20 - Loss: 705.2565


In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

# Tomar una muestra
image, target = dataset[10]

# DETR espera lista de imágenes
inputs = processor(images=[image], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-processing correcto
results = processor.post_process_object_detection(
    outputs,
    target_sizes=[(image.size[1], image.size[0])],  # (h, w)
    threshold=0.5
)[0]

# Plot
plt.figure(figsize=(8, 8))
plt.imshow(image)

for score, label, box in zip(
    results["scores"],
    results["labels"],
    results["boxes"]
):
    xmin, ymin, xmax, ymax = box.cpu().numpy()

    plt.gca().add_patch(
        plt.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            edgecolor="red",
            linewidth=2
        )
    )

    plt.text(
        xmin,
        ymin,
        model.config.id2label[label.item()],
        color="red",
        fontsize=8,
        bbox=dict(facecolor="yellow", alpha=0.5)
    )

plt.axis("off")
plt.show()